# Uber Pickup Analysis — New York City, April–September 2014

This notebook combines six monthly Uber pickup files from the FiveThirtyEight / NYC TLC FOIL dataset. It examines pickup volume by month, weekday, hour, dispatch base, and pickup coordinates.

Before running the notebook, place all six monthly CSV files in `data/raw/`. The notebook does not download data automatically.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

RAW_DIR = Path("data/raw")
MONTH_FILES = [
    "uber-raw-data-apr14.csv",
    "uber-raw-data-may14.csv",
    "uber-raw-data-jun14.csv",
    "uber-raw-data-jul14.csv",
    "uber-raw-data-aug14.csv",
    "uber-raw-data-sep14.csv",
]
missing = [name for name in MONTH_FILES if not (RAW_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        "Missing monthly CSVs. Run `python download_data.py` first. Missing: "
        + ", ".join(missing)
    )

## 1. Load and combine the monthly files

In [ ]:
monthly_frames = []
for filename in MONTH_FILES:
    month_df = pd.read_csv(RAW_DIR / filename)
    month_df["Date/Time"] = pd.to_datetime(
        month_df["Date/Time"], format="%m/%d/%Y %H:%M:%S", errors="raise"
    )
    monthly_frames.append(month_df)

uber = pd.concat(monthly_frames, ignore_index=True)
uber["Month"] = uber["Date/Time"].dt.month_name()
uber["Weekday"] = uber["Date/Time"].dt.day_name()
uber["Hour"] = uber["Date/Time"].dt.hour
uber["Date"] = uber["Date/Time"].dt.date

print(f"Rows: {len(uber):,}")
print(f"Date range: {uber['Date/Time'].min()} to {uber['Date/Time'].max()}")
print(f"Columns: {list(uber.columns)}")
uber.head()

## 2. Basic data checks

The source files contain pickup timestamps, coordinates, and a dispatch-base code. This section checks the combined table before summarizing it.

In [ ]:
uber.info()

quality = pd.DataFrame({
    "missing_values": uber.isna().sum(),
    "dtype": uber.dtypes.astype(str),
})
display(quality)
display(uber["Base"].value_counts().rename("pickup_count").to_frame())
print(f"Exact duplicate rows: {uber.duplicated().sum():,}")

## 3. Pickups by month

Monthly totals are raw counts. Since months contain different numbers of days, they are not daily averages.

In [ ]:
month_order = ["April", "May", "June", "July", "August", "September"]
monthly_counts = uber["Month"].value_counts().reindex(month_order)
display(monthly_counts.rename("pickups").to_frame())

growth_pct = (monthly_counts.iloc[-1] / monthly_counts.iloc[0] - 1) * 100
print(f"April to September change: {growth_pct:.2f}%")

ax = monthly_counts.plot(kind="bar", figsize=(9, 5), title="Uber pickups by month")
ax.set_xlabel("Month")
ax.set_ylabel("Pickups")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Pickups by weekday

These are total pickups across the full six-month period, not averages per occurrence of each weekday.

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_counts = uber["Weekday"].value_counts().reindex(weekday_order)
display(weekday_counts.rename("pickups").to_frame())

ax = weekday_counts.plot(kind="bar", figsize=(9, 5), title="Uber pickups by weekday")
ax.set_xlabel("Weekday")
ax.set_ylabel("Pickups")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 5. Pickups by hour

The hourly distribution shows when pickups were most and least frequent across the full period.

In [ ]:
hourly_counts = uber["Hour"].value_counts().sort_index().reindex(range(24), fill_value=0)
display(hourly_counts.rename("pickups").to_frame())
print(f"Busiest hour: {hourly_counts.idxmax():02d}:00 ({hourly_counts.max():,} pickups)")
print(f"Quietest hour: {hourly_counts.idxmin():02d}:00 ({hourly_counts.min():,} pickups)")

ax = hourly_counts.plot(kind="bar", figsize=(11, 5), title="Uber pickups by hour of day")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Pickups")
plt.tight_layout()
plt.show()

## 6. Monthly patterns by hour and weekday

In [ ]:
month_hour = uber.pivot_table(
    index="Month", columns="Hour", values="Base", aggfunc="size", fill_value=0
).reindex(month_order)
plt.figure(figsize=(14, 5))
sns.heatmap(month_hour, cmap="YlGnBu")
plt.title("Pickups by month and hour")
plt.xlabel("Hour")
plt.ylabel("Month")
plt.tight_layout()
plt.show()

month_weekday = uber.pivot_table(
    index="Month", columns="Weekday", values="Base", aggfunc="size", fill_value=0
).reindex(index=month_order, columns=weekday_order)
plt.figure(figsize=(10, 5))
sns.heatmap(month_weekday, annot=True, fmt=",", cmap="YlGnBu")
plt.title("Pickups by month and weekday")
plt.xlabel("Weekday")
plt.ylabel("Month")
plt.tight_layout()
plt.show()

## 7. Dispatch-base distribution

In [ ]:
base_counts = uber["Base"].value_counts()
base_share = (base_counts / len(uber) * 100).round(2)
base_summary = pd.DataFrame({"pickups": base_counts, "share_pct": base_share})
display(base_summary)

top_three_share = base_counts.nlargest(3).sum() / len(uber) * 100
print(f"Top three bases share: {top_three_share:.2f}%")

ax = base_counts.sort_values(ascending=False).plot(
    kind="bar", figsize=(9, 5), title="Pickups by dispatch base"
)
ax.set_xlabel("Base")
ax.set_ylabel("Pickups")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

month_base = uber.pivot_table(
    index="Month", columns="Base", values="Date", aggfunc="count", fill_value=0
).reindex(month_order)
plt.figure(figsize=(9, 5))
sns.heatmap(month_base, annot=True, fmt=",", cmap="YlGnBu")
plt.title("Monthly pickups by dispatch base")
plt.xlabel("Base")
plt.ylabel("Month")
plt.tight_layout()
plt.show()

## 8. Pickup coordinates

The source contains pickup latitude and longitude, but no borough label. The plot below samples points for readability; it does not assign official borough boundaries.

In [ ]:
# Keep coordinates in a broad NYC viewing window for this plot only.
# The full dataset remains unchanged for all count summaries above.
nyc_view = uber.loc[
    uber["Lon"].between(-74.3, -73.6) & uber["Lat"].between(40.45, 41.0),
    ["Lon", "Lat"],
]
plot_points = nyc_view.sample(n=min(100_000, len(nyc_view)), random_state=42)

plt.figure(figsize=(9, 9))
plt.scatter(plot_points["Lon"], plot_points["Lat"], s=0.15, alpha=0.25)
plt.title("Sample of Uber pickup coordinates in the NYC area")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

print(f"Points in viewing window: {len(nyc_view):,} of {len(uber):,} total pickups")

## 9. Summary

The analysis identifies differences in pickup volume across months, weekdays, hours, and dispatch bases. The busiest weekday and hour are based on total counts across April–September 2014. These descriptive patterns show when and where pickups were recorded; they do not explain the causes of demand or measure driver availability.

In [ ]:
summary = {
    "total_pickups": len(uber),
    "april_pickups": int(monthly_counts.loc["April"]),
    "september_pickups": int(monthly_counts.loc["September"]),
    "april_to_september_growth_pct": round(float(growth_pct), 2),
    "busiest_weekday": weekday_counts.idxmax(),
    "busiest_weekday_pickups": int(weekday_counts.max()),
    "busiest_hour": int(hourly_counts.idxmax()),
    "busiest_hour_pickups": int(hourly_counts.max()),
    "quietest_hour": int(hourly_counts.idxmin()),
    "quietest_hour_pickups": int(hourly_counts.min()),
    "top_three_base_share_pct": round(float(top_three_share), 2),
}
pd.Series(summary, name="value").to_frame()